# 零、总述

在上个笔记本中，我们了解了分词和词嵌入的基本概念，现在我们要深入研究语言模型的工作原理了，本节主要讨论 Transformer LLM 的主要工作原理，我们将重点关注文本生成模型，以便更深入地理解生成式 LLM

# 一、Transformer 模型概述

## 1.1 使用 Transformer LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 加载模型和分词器
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False
)

接下来，我们创建一个预测 token 的 Pipeline


此处的 Pipeline 对象是一个封装了 **输入预处理、模型推理和输出后推理** 的可调用对象，它更像是一个 任务执行器/模型调用器

In [ ]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False
)

让我们使用 Pipeline(generator) 来续写如下文段

In [ ]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

output = generator(prompt)

print(output[0]['generated_text'])

我们看一下 `microsoft/Phi-3-mini-4k-instruct` 模型的架构

In [ ]:
print(model)

## 1.2 前向传播的组成

## 1.3 从概率分布中选择单个词元(采样/编码)

In [ ]:
prompt = "The capital of France is"

input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

# Get the output of the model before the lm_head
model_output = model.model(input_ids)

# Get the output of the lm_head
lm_head_output = model.lm_head(model_output[0])

In [ ]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

In [ ]:
model_output[0].shape

In [ ]:
lm_head_output.shape

## 1.4 并行词元处理和上下文长度

## 1.5 通过 KV-Cache 加速生成过程

In [ ]:
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# 对 prompt 进行分词
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

使用 KV-Cache

In [ ]:
%%timeit -n 1
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=100,
    use_cache=True
)

不使用 KV-Cache

In [ ]:
%%timeit -n 1
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=100,
    use_cache=False
)

## 1.6 Transformer 块的内部结构

# 二、Transformer 架构的最新改进

## 2.1 更高效的注意力机制

## 2.2 Transformer 块

## 2.3 位置嵌入：RoPE

## 2.4 其他架构实验与改进

# 三、小结